# MedGemma IE few-shot inference with completion-style prompting

This notebook runs **base MedGemma** without LoRA fine-tuning, using few-shot examples taken from the training set.

It mirrors the **completion SFT format**:

```text
<prompt: instruction + report + model turn>
<completion: JSON>
```

For few-shot inference, the prompt contains several complete examples:

```text
Example 1 prompt -> Example 1 JSON
Example 2 prompt -> Example 2 JSON
...
Target report prompt -> model generates JSON
```

To be used as a baseline against completion-SFT.

In [1]:
import os
import json
import math
import random
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
from collections import Counter, defaultdict

import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM

## Configuration

Set `MODEL_ID` to either the Hugging Face model id or local absolute path.

`TRAIN_FILE` should be the same JSONL used for SFT, with one example per line:

```json
{"report": "...", "label": {"echantillon": "...", ...}}
```

In [2]:
MODEL_ID = "google/medgemma-27b-text-it"

# for few shots use the training
TRAIN_FILE = "training_data/train_normalized.jsonl"  
VALID_FILE = None  


N_SHOTS = 0


OUTPUT_PREFIX = "base_medgemma_fewshot_completion"
if N_SHOTS==0:
    OUTPUT_PREFIX= "base_medgemma_completion"

MAX_NEW_TOKENS = 256
SEED = 42

torch.manual_seed(SEED)
random.seed(SEED)

## Prompt and schema helpers

This uses the same semantic extraction prompt as your completion SFT setup. The few-shot examples are formatted as completed model turns, but no LoRA adapter is loaded.

In [3]:
FIELD_ORDER = [
    "echantillon",
    "numero_gang_prel",
    "numero_gang_att",
    "taille",
    "diagnostic",
    "grade",
    "emboles",
    "ki67",
    "re",
    "rp",
    "her2",
    "marges_saines",
    "rupture_capsulaire",
]

NUMERIC_FIELDS = {"numero_gang_prel", "numero_gang_att", "taille", "ki67"}
CATEGORICAL_FIELDS = [f for f in FIELD_ORDER if f not in NUMERIC_FIELDS]

SYSTEM_PROMPT = """
Tu es un système d'extraction d'information à partir de comptes rendus anatomopathologiques de cancer du sein.

Retourne uniquement un objet JSON valide.
N'invente aucune information.

Si une information est absente, ambiguë ou non déductible:
- utilise "unknown" pour les variables catégorielles
- utilise null pour les variables numériques.

Variables à extraire:
- echantillon: mammectomie, tumorectomie, curage ganglionnaire, biopsie, exérèse chirurgicale, unknown
- numero_gang_prel: nombre de ganglions prélevés/examinés, integer ou null
- numero_gang_att: nombre de ganglions atteints/envahis, integer ou null
- taille: taille tumorale maximale en mm, integer ou null
- diagnostic: histologie principale, string
- grade: SBR1, SBR2, SBR3, unknown
- emboles: oui, non, unknown
- ki67: pourcentage Ki67, integer ou null
- re: positifs, négatifs, unknown
- rp: positifs, négatifs, unknown
- her2: négatif, douteux, positif, unknown
- marges_saines: oui, non, unknown
- rupture_capsulaire: oui, non, unknown

Règles:
- Convertir les cm en mm pour taille.
- HER2 0 ou 1+ = négatif.
- HER2 2+ = douteux.
- HER2 3+ ou amplification FISH/CISH/SISH = positif.
- RE/RP positifs si expression positive, même faible.
- Marges R0/non envahies/saines = oui.
- Marges atteintes/R1/envahies = non.
""".strip()


def build_user_message(report: str) -> str:
    return (
        SYSTEM_PROMPT
        + '\n\nCompte rendu:\n"""\n'
        + str(report)
        + '\n"""\n\nRetourne uniquement le JSON.'
    )




def dumps_label(label: Dict[str, Any]) -> str:
    ordered = {}
    for key in FIELD_ORDER:
        value = label.get(key)
        if key in NUMERIC_FIELDS:
            if value == "unknown" or (isinstance(value, float) and math.isnan(value)):
                value = None
            elif value is not None:
                try:
                    value = int(value)
                except (TypeError, ValueError):
                    pass
        else:
            if value is None or (isinstance(value, float) and math.isnan(value)):
                value = "unknown"
        ordered[key] = value
    return json.dumps(ordered, ensure_ascii=False, separators=(",", ":"))

In [1]:
df_trainingex=pd.read_json(TRAIN_FILE,lines=True)
def parse_label(label: Any) -> Dict[str, Any]:
    if isinstance(label, str):
        label = json.loads(label)
    if not isinstance(label, dict):
        raise ValueError(f"label must be a dict or JSON string, got {type(label)}")
    return label

def make_flat_train_df(train_df):
    rows = []

    for idx, row in train_df.iterrows():
        label = parse_label(row["label"])

        flat_row = {
            "row_id": idx,
            "report": row["report"],
        }

        for field in FIELD_ORDER:
            flat_row[field] = label.get(field)

        rows.append(flat_row)

    return pd.DataFrame(rows)

flat_train_df = make_flat_train_df(df_trainingex)
flat_train_df['grade'].unique()


## Load training examples for few-shot prompts

`train_examples` is a list of examples from training JSONL. Each selected example becomes one demonstration in the prompt.

In [2]:
def load_jsonl_examples(path: str, max_examples: Optional[int] = None) -> List[Dict[str, Any]]:
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_examples is not None and i >= max_examples:
                break
            ex = json.loads(line)
            if "report" not in ex or "label" not in ex:
                raise ValueError(f"Each JSONL row must have report and label keys. Bad row index: {i}")
            examples.append(ex)
    return examples


def validate_jsonl(path: str) -> Dict[str, Any]:
    errors = []
    missing = Counter()
    n = 0
    with open(path, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f, start=1):
            n += 1
            try:
                ex = json.loads(line)
            except json.JSONDecodeError as exc:
                errors.append(f"line {line_idx}: invalid JSON: {exc}")
                continue
            if not isinstance(ex.get("report"), str):
                errors.append(f"line {line_idx}: missing or non-string report")
            try:
                label = parse_label(ex.get("label"))
            except Exception as exc:
                errors.append(f"line {line_idx}: invalid label: {exc}")
                continue
            for field in FIELD_ORDER:
                if field not in label:
                    missing[field] += 1
    return {
        "n_examples": n,
        "n_errors": len(errors),
        "errors_first_20": errors[:20],
        "missing_fields": dict(missing),
    }

print("Train check:", validate_jsonl(TRAIN_FILE))
if VALID_FILE is not None:
    print("Valid check:", validate_jsonl(VALID_FILE))

train_examples = load_jsonl_examples(TRAIN_FILE)
print("Loaded train examples:", len(train_examples))
print("First example keys:", train_examples[0].keys())

## Load base MedGemma

This loads the base model only. No LoRA adapter is loaded.

Use `use_fast=False` to avoid tokenizer backend issues in environments without `sentencepiece`/`tiktoken` support.

In [6]:
def resolve_model_name_or_path(model_name_or_path: str) -> str:
    p = Path(model_name_or_path).expanduser()
    if p.exists():
        return str(p.resolve())
    if model_name_or_path.count("/") <= 1 and not model_name_or_path.startswith("."):
        return model_name_or_path
    raise FileNotFoundError(
        f"Local model path does not exist: {p.resolve()}\n"
        "Use an absolute path, fix the relative path, or use the HF repo id."
    )


def load_base_model_and_tokenizer(model_id: str = MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"):
    resolved = resolve_model_name_or_path(model_id)

    tok = AutoTokenizer.from_pretrained(
        resolved,
        trust_remote_code=True,
        use_fast=False,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        resolved,
        torch_dtype=torch_dtype,
        device_map=device_map,
        trust_remote_code=True,
    )
    model.eval()
    model.config.use_cache = True
    return model, tok

model_base, tokenizer_base = load_base_model_and_tokenizer(MODEL_ID)
print("Loaded base model and tokenizer")
print("pad_token_id:", tokenizer_base.pad_token_id)
print("eos_token_id:", tokenizer_base.eos_token_id)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

Loaded base model and tokenizer
pad_token_id: 0
eos_token_id: 1


## Few-shot selection

For a fair baseline, keep the same `shot_indices` for all validation/test rows.

Start with 3 shots. 

In [3]:
def examples_to_train_df(train_examples: List[Dict[str, Any]]) -> pd.DataFrame:
    """Create a dataframe with one row per training example.

    The dataframe index is the same as the train_examples list position, so
    train_df.index values can be used directly as few-shot indices.
    """
    rows = []
    for i, ex in enumerate(train_examples):
        label = parse_label(ex["label"])
        row = {"row_id": i, "report": ex["report"]}
        for field in FIELD_ORDER:
            value = label.get(field)
            if value == "unknown" and field in NUMERIC_FIELDS:
                value = None
            row[field] = value
        rows.append(row)

    train_df = pd.DataFrame(rows)
    train_df.index = train_df["row_id"].astype(int)

    # Make numeric fields numeric so comparisons such as > 0 work reliably.
    for field in NUMERIC_FIELDS:
        train_df[field] = pd.to_numeric(train_df[field], errors="coerce")

    return train_df



from typing import Dict, List
import pandas as pd


def find_candidates(train_df: pd.DataFrame) -> Dict[str, List[int]]:
    candidates = {}

    def s(col):
        return train_df[col].astype(str).str.strip().str.lower()

    # 1. Biopsie with in situ:
    #    diagnostic 'carcinome intracanalaire non infiltrant'
    mask_biopsie_absent = (
        s("echantillon").eq("biopsie")
        & (
            train_df["diagnostic"].isna()
            | s("diagnostic").isin(["", "unknown", "nan", "none", "null", 'carcinome intracanalaire non infiltrant'])
        )
        & train_df["numero_gang_prel"].isna()
        & train_df["numero_gang_att"].isna()
        & train_df["taille"].isna()
        & train_df["ki67"].isna()
        & s("grade").eq("unknown")
    )

    candidates["biopsie_absent_information"] = (
        train_df[mask_biopsie_absent].index.astype(int).tolist()[:10]
    )

    # 2. Tumorectomie with ganglions:
    #    known sampled/examined nodes and known positive/affected nodes
    mask_tumorectomie_ganglions = (
        s("echantillon").eq("tumorectomie")
        & train_df["numero_gang_prel"].notna()
        & train_df["numero_gang_att"].notna()
        & (train_df["numero_gang_prel"] > 0)
    )

    candidates["tumorectomie_with_ganglions"] = (
        train_df[mask_tumorectomie_ganglions].index.astype(int).tolist()[:10]
    )

    # 3. Mammectomie:
    #    preferably with missing ganglions / missing Ki67 / unknown grade,
    #    to keep the example useful for absent values too
    mask_mammectomie = (
        s("echantillon").eq("mammectomie")
        & train_df["numero_gang_prel"].isna()
        & train_df["numero_gang_att"].isna()
        & train_df["ki67"].isna()
        & s("grade").eq("unknown")
    )

    candidates["mammectomie_missing_values"] = (
        train_df[mask_mammectomie].index.astype(int).tolist()[:10]
    )

    return candidates

def choose_representative_shots(
    train_examples: List[Dict[str, Any]],
    n_shots: int = 3,
    seed: int = 42,
    category_order: Optional[List[str]] = None,
) -> List[int]:
    """Choose fixed clinically representative shots.

    It tries to pick one example from each category in category_order.
    If a category has no candidate, it skips it. If fewer than n_shots are
    found, it fills the rest with random unused examples.
    """
    rng = random.Random(seed)
    train_df = examples_to_train_df(train_examples)
    candidates = find_candidates(train_df)

    if category_order is None:
        category_order = [
            "mammectomie_missing_values",
            "biopsie_absent_information",
            "tumorectomie_with_ganglions",
            "her2_douteux",
            "marges_non_saines",
        ]

    selected = []
    selected_from = []

    for category in category_order:
        available = [idx for idx in candidates.get(category, []) if idx not in selected]
        if not available:
            continue
        selected.append(rng.choice(available))
        selected_from.append(category)
        if len(selected) >= n_shots:
            break

    if len(selected) < n_shots:
        remaining = [i for i in range(len(train_examples)) if i not in selected]
        fill = rng.sample(remaining, k=min(n_shots - len(selected), len(remaining)))
        selected.extend(fill)
        selected_from.extend(["random_fallback"] * len(fill))

    print("Candidate counts:")
    for name, idxs in candidates.items():
        print(f"- {name}: {len(idxs)} shown / first candidates: {idxs[:5]}")

    print("\nSelected shots:")
    for idx, source in zip(selected, selected_from):
        print(f"- {idx} from {source}: {dumps_label(parse_label(train_examples[idx]['label']))}")

    return selected


# Option A: fixed manual indices (or random but they are not good any)
# shot_indices = [0, 1, 2]

# Option B: clinically representative fixed shots
shot_indices = choose_representative_shots(
    train_examples,
    n_shots=N_SHOTS,
    seed=SEED,
)
if N_SHOTS==0: 
    shot_indices=[]
    

print("\nshot_indices:", shot_indices)


## Build completion-style few-shot prompt

Each few-shot example is rendered as a completed chat turn:

```text
<start_of_turn>user
instruction + report
<end_of_turn>
<start_of_turn>model
JSON
<end_of_turn>
```

Then the target report is appended as the final user turn, followed by the model generation prompt.

In [9]:
def build_few_shot_messages_jsonl(
    report: str,
    train_examples: List[Dict[str, Any]],
    shot_indices: List[int],
) -> List[Dict[str, str]]:
    messages = []

    for idx in shot_indices:
        shot = train_examples[idx]
        messages.append({
            "role": "user",
            "content": build_user_message(shot["report"]),
        })
        # Gemma renders assistant-like turns as <start_of_turn>model.
        messages.append({
            "role": "model",
            "content": dumps_label(parse_label(shot["label"])),
        })

    messages.append({
        "role": "user",
        "content": build_user_message(report),
    })

    return messages


def render_few_shot_prompt(report: str, tokenizer, train_examples, shot_indices) -> str:
    messages = build_few_shot_messages_jsonl(report, train_examples, shot_indices)
    # print(messages)
    # print("-----------------")
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

preview_prompt = render_few_shot_prompt(
    report=train_examples[1]["report"],
    tokenizer=tokenizer_base,
    train_examples=train_examples,
    shot_indices=shot_indices,
)
print(preview_prompt[:3000])
print("\n--- prompt token length ---")
print(len(tokenizer_base(preview_prompt, add_special_tokens=False)["input_ids"]))

<bos><start_of_turn>user
Tu es un système d'extraction d'information à partir de comptes rendus anatomopathologiques de cancer du sein.

Retourne uniquement un objet JSON valide.
N'invente aucune information.

Si une information est absente, ambiguë ou non déductible:
- utilise "unknown" pour les variables catégorielles
- utilise null pour les variables numériques.

Variables à extraire:
- echantillon: mammectomie, tumorectomie, curage ganglionnaire, biopsie, exérèse chirurgicale, unknown
- numero_gang_prel: nombre de ganglions prélevés/examinés, integer ou null
- numero_gang_att: nombre de ganglions atteints/envahis, integer ou null
- taille: taille tumorale maximale en mm, integer ou null
- diagnostic: histologie principale, string
- grade: SBR1, SBR2, SBR3, unknown
- emboles: oui, non, unknown
- ki67: pourcentage Ki67, integer ou null
- re: positifs, négatifs, unknown
- rp: positifs, négatifs, unknown
- her2: négatif, douteux, positif, unknown
- marges_saines: oui, non, unknown
- ru

## JSON parsing and generation

In [10]:
def extract_first_json_object(text: str) -> Optional[Dict[str, Any]]:
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    in_string = False
    escape = False

    for idx in range(start, len(text)):
        ch = text[idx]
        if escape:
            escape = False
            continue
        if ch == "\\":
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    obj = json.loads(text[start:idx + 1])
                    return obj if isinstance(obj, dict) else None
                except json.JSONDecodeError:
                    return None
    return None


@torch.inference_mode()
def generate_extraction_base_fewshot_completion(
    report: str,
    model,
    tokenizer,
    train_examples: List[Dict[str, Any]],
    shot_indices: List[int],
    max_new_tokens: int = 256,
) -> Tuple[str, Optional[Dict[str, Any]]]:
    prompt = render_few_shot_prompt(
        report=report,
        tokenizer=tokenizer,
        train_examples=train_examples,
        shot_indices=shot_indices,
    )
    # print(prompt)

    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return raw_text, extract_first_json_object(raw_text)

## Single-example test

In [4]:
example_report = train_examples[15]["report"]

raw_text, pred_json = generate_extraction_base_fewshot_completion(
    report=example_report,
    model=model_base,
    tokenizer=tokenizer_base,
    train_examples=train_examples,
    shot_indices=shot_indices,
    max_new_tokens=MAX_NEW_TOKENS,
)

print("RAW OUTPUT:")
print(raw_text)
print("\nPARSED JSON:")
print(json.dumps(pred_json, ensure_ascii=False, indent=2))

## Run few-shot inference from a dataframe


### load gold

In [12]:
import pandas as pd
df= pd.read_json('./data/gold_data.json')

df.columns

Index(['content', 'echantillon', 'numero_gang_prel', 'taille', 'diagnostic',
       'grade', 'emboles', 'numero_gang_att', 'ki67', 're', 'rp', 'her2',
       'marges_saines', 'rupture_capsulaire'],
      dtype='object')

In [13]:
def clean_value(x):
    if pd.isna(x):
        return None
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        if float(x).is_integer():
            return int(x)
        return float(x)
    return x


In [14]:



@torch.inference_mode()
def run_fewshot_inference_dataframe(
    df: pd.DataFrame,
    model,
    tokenizer,
    train_examples: List[Dict[str, Any]],
    shot_indices: List[int],
    report_col: str = "content",
    max_new_tokens: int = 256,
    output_prefix: str = OUTPUT_PREFIX,
    save_every: int = 25,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:

    import os
    import json

    missing_cols = [c for c in FIELD_ORDER if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing gold columns in dataframe: {missing_cols}")

    if report_col not in df.columns:
        raise ValueError(f"Missing report column in dataframe: {report_col}")

    tmp_json_path = f"{output_prefix}_tmp_predictions.json"
    final_json_path = f"{output_prefix}_results.json"
    final_csv_path = f"{output_prefix}_results.csv"

    results = []

    for i, report in enumerate(
        tqdm(df[report_col], total=len(df), desc="Few-shot dataframe inference")
    ):
        raw_text, pred_json = generate_extraction_base_fewshot_completion(
            report=str(report),
            model=model,
            tokenizer=tokenizer,
            train_examples=train_examples,
            shot_indices=shot_indices,
            max_new_tokens=max_new_tokens,
        )

        gold_variables = {
            field: clean_value(df.iloc[i][field])
            for field in FIELD_ORDER
        }

        results.append({
            "row_id": i,
            "report": str(report),
            "gold_variables": gold_variables,
            "prediction": pred_json,
            "raw_text": raw_text,
            "shot_indices": shot_indices,
        })

        # temp checkpoint
        if save_every is not None and save_every > 0 and (i + 1) % save_every == 0:
            with open(tmp_json_path, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

            print(f"Saved temporary predictions: {tmp_json_path} ({i + 1}/{len(df)})")


    # final JSON
    with open(final_json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    # final CSV
    rows = []
    for item in results:
        row = {
            "row_id": item["row_id"],
            "report": item["report"],
            "raw_text": item["raw_text"],
            "valid_json": item["prediction"] is not None,
            "shot_indices": ",".join(map(str, item.get("shot_indices", []))),
        }

        pred = item["prediction"] or {}
        for field in FIELD_ORDER:
            row[f"pred_{field}"] = pred.get(field)

        gold = item["gold_variables"] or {}
        for field in FIELD_ORDER:
            row[f"gold_{field}"] = gold.get(field)

        rows.append(row)

    results_df = pd.DataFrame(rows)
    results_df.to_csv(final_csv_path, index=False, encoding="utf-8-sig")

    print(f"Saved temporary predictions: {tmp_json_path}")
    print(f"Saved final results JSON: {final_json_path}")
    print(f"Saved final results CSV: {final_csv_path}")

    return results


# BATCHING

In [15]:
from typing import List, Dict, Any, Tuple
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import torch
import json


def clean_value(x):
    if pd.isna(x):
        return None
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        if float(x).is_integer():
            return int(x)
        return float(x)
    return x


def build_fewshot_completion_prompt(
    report: str,
    tokenizer,
    train_examples: List[Dict[str, Any]],
    shot_indices: List[int],
) -> str:
    messages = []

    for idx in shot_indices:
        ex = train_examples[idx]

        messages.append({
            "role": "user",
            "content": build_user_message(ex["report"]),
        })

        messages.append({
            "role": "model",
            "content": dumps_label(parse_label(ex["label"])),
        })

    messages.append({
        "role": "user",
        "content": build_user_message(report),
    })

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


@torch.inference_mode()
def generate_extraction_base_fewshot_completion_batch(
    reports: List[str],
    model,
    tokenizer,
    train_examples: List[Dict[str, Any]],
    shot_indices: List[int],
    max_new_tokens: int = 256,
) -> List[Tuple[str, Any]]:

    prompts = [
        build_fewshot_completion_prompt(
            report=str(report),
            tokenizer=tokenizer,
            train_examples=train_examples,
            shot_indices=shot_indices,
        )
        for report in reports
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False,
    ).to(model.device)

    input_lengths = inputs["attention_mask"].sum(dim=1).tolist()

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    outputs = []

    for j in range(len(reports)):
        generated_ids = output_ids[j][int(input_lengths[j]):]
        raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        pred_json = extract_first_json_object(raw_text)
        outputs.append((raw_text, pred_json))

    return outputs

In [16]:
@torch.inference_mode()
def run_fewshot_inference_dataframe_batched(
    df: pd.DataFrame,
    model,
    tokenizer,
    train_examples: List[Dict[str, Any]],
    shot_indices: List[int],
    report_col: str = "content",
    max_new_tokens: int = 256,
    output_prefix: str = OUTPUT_PREFIX,
    save_every_batches: int = 5,
    batch_size: int = 4,
) -> List[Dict[str, Any]]:

    missing_cols = [c for c in FIELD_ORDER if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing gold columns in dataframe: {missing_cols}")

    if report_col not in df.columns:
        raise ValueError(f"Missing report column in dataframe: {report_col}")

    tmp_json_path = f"{output_prefix}_tmp_predictions2105.json"
    final_json_path = f"{output_prefix}_results2105.json"
    final_csv_path = f"{output_prefix}_results2105.csv"

    results = []

    n = len(df)

    for batch_start in tqdm(
        range(0, n, batch_size),
        total=(n + batch_size - 1) // batch_size,
        desc="Few-shot batched dataframe inference",
    ):
        batch_end = min(batch_start + batch_size, n)

        batch_reports = [
            str(x)
            for x in df.iloc[batch_start:batch_end][report_col].tolist()
        ]

        batch_outputs = generate_extraction_base_fewshot_completion_batch(
            reports=batch_reports,
            model=model,
            tokenizer=tokenizer,
            train_examples=train_examples,
            shot_indices=shot_indices,
            max_new_tokens=max_new_tokens,
        )

        for offset, (raw_text, pred_json) in enumerate(batch_outputs):
            i = batch_start + offset

            gold_variables = {
                field: clean_value(df.iloc[i][field])
                for field in FIELD_ORDER
            }

            results.append({
                "row_id": i,
                "original_index": clean_value(df.index[i]),
                "report": str(df.iloc[i][report_col]),
                "gold_variables": gold_variables,
                "prediction": pred_json,
                "raw_text": raw_text,
                "shot_indices": shot_indices,
            })

        batch_number = batch_start // batch_size + 1

        if (
            save_every_batches is not None
            and save_every_batches > 0
            and batch_number % save_every_batches == 0
        ):
            with open(tmp_json_path, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

            print(
                f"Saved temporary predictions: "
                f"{tmp_json_path} ({len(results)}/{n})"
            )

    with open(final_json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    rows = []

    for item in results:
        row = {
            "row_id": item["row_id"],
            "original_index": item["original_index"],
            "report": item["report"],
            "raw_text": item["raw_text"],
            "valid_json": item["prediction"] is not None,
            "shot_indices": ",".join(map(str, item.get("shot_indices", []))),
        }

        pred = item["prediction"] or {}
        for field in FIELD_ORDER:
            row[f"pred_{field}"] = pred.get(field)

        gold = item["gold_variables"] or {}
        for field in FIELD_ORDER:
            row[f"gold_{field}"] = gold.get(field)

        rows.append(row)

    results_df = pd.DataFrame(rows)
    results_df.to_csv(final_csv_path, index=False, encoding="utf-8-sig")

    print(f"Saved final results JSON: {final_json_path}")
    print(f"Saved final results CSV: {final_csv_path}")

    return results

In [5]:
tokenizer_base.padding_side = "left"

results_base_fewshot = run_fewshot_inference_dataframe_batched(
    df=df,
    model=model_base,
    tokenizer=tokenizer_base,
    train_examples=train_examples,
    shot_indices=shot_indices,
    report_col="content",
    max_new_tokens=256,
    output_prefix="outputInf/"+ OUTPUT_PREFIX,
    batch_size=4,
    save_every_batches=5,
)

